## Lab: Building a Trigram Language Model

#### What we will build

A small language model that learns from a short conversation, then **predicts the next word**, just like a phone keyboard.

#### The idea in one line

> The model reads the training text, **counts** which word comes after each **pair of words**, and turns those counts into **probabilities**.

#### Our training text

A short interview between an officer and a suspect (12 lines):

```
officer: where were you last night
suspect: I was at home last night
officer: who was with you
...
```

#### What we will do

| Phase | Step | What happens |
| --- | --- | --- |
| **Training** | 1. Tokenize | Split each line into words and add two `<s>` and two `</s>` |
| | 2. Count words | How many times does each word appear? |
| | 3. Count triples | How many times does each triple of words appear? |
| | 4. Probabilities | P(next \| previous two) = C(w1 w2 w3) / C(w1 w2) |
| **Using the model** | 5. Predict | Type two words, and the model suggests the next one |
| **Testing** | 6–10 | Score new sentences, find the zero problem, and fix it |

In [ ]:
text="""
officer: where were you last night
suspect: I was at home last night
officer: who was with you
suspect: I was alone at home
officer: you were not at home
suspect: I was at home all night
officer: did you see the car
suspect: I did not see the car
officer: where is the car now
suspect: I do not know
officer: you know where the car is
suspect: I do not know where it is
"""

In [ ]:
print(text)

In [ ]:
import re
from collections import Counter

In [ ]:
sentences = text.strip().split("\n")
sentence_tokens = [["<s>", "<s>"] + re.findall(r'\w+', sentence.lower()) + ["</s>", "</s>"] for sentence in sentences]
for sentence, token_list in zip(sentences, sentence_tokens):
    print(f"Sentence: {sentence}")
    print(f"Tokens: {token_list}")
    print()

In [ ]:
def get_frequency(tokens):
    frequency = dict()
    for token_list in tokens:
        for w, f in Counter(token_list).items():
            frequency[w] = frequency.get(w, 0) + f
    return frequency


frequency = get_frequency(sentence_tokens)

for token, freq in frequency.items():
    print(f"Token: {token:8}, Frequency: {freq}")
print(f"Total unique tokens - vocabulary size: {len(frequency)}")

In [ ]:
print(text)

In [ ]:
def get_trigrams(tokens):
    trigrams = Counter()

    for token_list in tokens:
        trigrams.update(
            (token_list[i], token_list[i + 1], token_list[i + 2])
            for i in range(len(token_list) - 2)
        )
    return trigrams

trigrams = get_trigrams(sentence_tokens)
print(f"Total unique trigrams: {len(trigrams)}")
for trigram, counts in trigrams.items():
    print(f"Trigram: {str(trigram):35}, Count: {counts}")

In [ ]:
def get_trigrams2(tokens):
    trigrams = []
    for token_list in tokens:
        trigrams.extend([(token_list[i], token_list[i + 1], token_list[i + 2]) for i in range(len(token_list) - 2)])
    return trigrams

trigrams2 = get_trigrams2(sentence_tokens)

print(f"Total trigrams: {len(trigrams)}")
print(f"Total unique trigrams: {len(set(trigrams))}")
print("------------------------------")

unique_trigrams = set(trigrams2)
i = 0
for trigram in unique_trigrams:
    count = trigrams2.count(trigram)
    print(f"Trigram: {trigram}, Count: {count}")
    i += 1
    print(f"Total trigrams processed: {i}")

In [ ]:
# Count how often each two-word context appears: C(w1, w2)
context_counts = Counter()
for (w1, w2, w3), count in trigrams.items():
    context_counts[(w1, w2)] += count

trigram_probabilities = {trigram: count / context_counts[trigram[:2]] for trigram, count in trigrams.items()}
for trigram, prob in trigram_probabilities.items():
    print(f"Trigram: {str(trigram):35}, Probability: {prob:.4f}")

### testing our trigram model

In [ ]:
def predict_next(w1, w2):
    context = (w1, w2)

    # Case 1: the word pair never appeared in training
    if context not in context_counts:
        print(f"'{w1} {w2}': Unknown context: no prediction.")
        return

    # Collect every trigram whose FIRST TWO words are the input pair
    candidates = {w3: p for (a, b, w3), p in trigram_probabilities.items() if (a, b) == context}

    # Case 2: nothing ever comes after this pair
    if not candidates:
        print(f"'{w1} {w2}': No next word.")
        return

    # Sort from most likely to least likely
    ordered = sorted(candidates.items(), key=lambda item: item[1], reverse=True)

    print(f"After '{w1} {w2}':")
    for next_word, prob in ordered:
        print(f"    {next_word:6} {prob:.2f}")
    print(f"    sum = {sum(candidates.values()):.2f}")


# DEMO: run this cell and you get output immediately, no typing needed
predict_next("i", "was")

In [ ]:
# OPTIONAL: type your own two words on ONE line, separated by a space
pair = input("Enter two words (e.g. i was): ").strip().lower().split()

if len(pair) != 2:
    print("Please enter exactly two words separated by a space.")
else:
    predict_next(pair[0], pair[1])

In [ ]:
def suggestions(w1, w2):
    context = (w1, w2)
    candidates = {w3: p for (a, b, w3), p in trigram_probabilities.items() if (a, b) == context}
    return sorted(candidates.items(), key=lambda item: item[1], reverse=True)


def type_with_suggestions():
    END_MARKS = {".", "?", "!"}
    sentence = []

    # 1. First two words, typed on ONE line (must be in the vocabulary)
    pair = input("Type two starting words (e.g. i was): ").strip().lower().split()
    while (len(pair) != 2 or
           any(w not in frequency or w in ("<s>", "</s>") for w in pair)):
        pair = input("Enter two known words separated by a space: ").strip().lower().split()
    w1, w2 = pair
    sentence.extend([w1, w2])

    # 2. Keep suggesting the next word until the user ends the sentence
    while True:
        print("\nSentence so far:", " ".join(sentence))
        options = suggestions(w1, w2)

        if not options:
            print("The model has no next word after this pair.")
            break

        for i, (next_word, prob) in enumerate(options, start=1):
            print(f"  {i}. {next_word:6} {prob:.2f}")
        choice = input("Choose a number (or type . ? ! to finish): ").strip()

        if choice in END_MARKS:            # the user ends the sentence
            sentence.append(choice)
            break
        if not choice.isdigit() or not 1 <= int(choice) <= len(options):
            print("Please type a number from the list, or . ? !")
            continue

        word = options[int(choice) - 1][0]
        if word == "</s>":                 # the user picked the model's end symbol
            print("The model chose the end of the sentence.")
            break
        sentence.append(word)

        # Slide the two-word context window forward
        w1, w2 = w2, word

    print("\nFinal sentence:", " ".join(sentence))


type_with_suggestions()